In [1]:
import torch
import torch.nn.functional as F
import numpy as np
from torch import nn
from tqdm import tqdm


In [2]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size= 5, embeddings_dim = 8):
        super().__init__()
        self.fc1 = nn.Linear(vocab_size, embeddings_dim, bias=False)
        self.out = nn.Linear(embeddings_dim, vocab_size, bias=False)

    def forward(self, x):
        x = self.fc1(x)
        x = self.out(x)
        return x


In [3]:
def tokenize(text: str):
    tokens = np.array(text.lower().split()).reshape(-1,1)
    return tokens

In [4]:
def encoding(tokens):
    _, ids = np.unique(tokens, return_inverse=True)
    return ids.ravel()

In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [6]:
phrase = encoding(tokenize("oi eu sou o thalles"))
print(phrase)

[2 0 3 1 4]


In [7]:
with open("data/text8") as f:
    text = f.read()

n_tokens = 1_000_00_0
corpus = text.split()[:n_tokens]
print(len(corpus), len(set(corpus)))

1000000 52754


In [8]:
tokens = tokenize(" ".join(corpus))
encoded = encoding(tokens)
print(tokens.shape, encoded.shape)

(1000000, 1) (1000000,)


In [9]:
model = SkipGram(vocab_size=int(encoded.max()) + 1, embeddings_dim=100).to(device)
print(model)

SkipGram(
  (fc1): Linear(in_features=52754, out_features=100, bias=False)
  (out): Linear(in_features=100, out_features=52754, bias=False)
)


In [10]:
ids = torch.tensor(encoded, device=device)

In [11]:
window = 2
centers, contexts = [], []
for i in range(len(tokens)):
    for j in range(max(0, i - window), min(len(tokens), i + window + 1)):
        if j != i:
            centers.append(i)
            contexts.append(j)


In [12]:
print(len(centers))

3999994


In [13]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.025)

n_epochs = 5
n_pairs = len(centers)
scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1.0, end_factor=1e-4, total_iters=n_epochs * n_pairs
)

centers_t = torch.tensor(centers, device=device)
contexts_t = torch.tensor(contexts, device=device)
vocab_size = model.fc1.in_features

for epoch in range(n_epochs):
    perm = torch.randperm(n_pairs, device=device)
    running = 0.0
    pbar = tqdm(range(n_pairs), desc=f"epoch {epoch}")
    for step in pbar:
        k = perm[step]
        x = F.one_hot(ids[centers_t[k]], vocab_size).float().unsqueeze(0)
        y = ids[contexts_t[k]].unsqueeze(0)

        loss = loss_fn(model(x), y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        running += loss.item()
        if (step + 1) % 1000 == 0:
            pbar.set_postfix(loss=running / 1000)
            running = 0.0


epoch 4: 100%|██████████| 3999994/3999994 [1:08:14<00:00, 976.84it/s, loss=6.67]


In [14]:
vocab = np.unique(tokens)
assert len(vocab) == model.fc1.in_features
word2idx = {w: i for i, w in enumerate(vocab)}
emb = F.normalize(model.fc1.weight.detach().T, dim=1)
print(emb.shape)

torch.Size([52754, 100])


In [15]:
from collections import Counter
freq = Counter(tokens.ravel().tolist())
print(freq.most_common(50))

[('the', 62827), ('of', 36789), ('and', 25238), ('one', 24679), ('in', 22502), ('a', 18620), ('to', 18504), ('zero', 14349), ('nine', 14056), ('is', 11094), ('two', 10968), ('as', 7737), ('eight', 7708), ('three', 7049), ('was', 6892), ('by', 6796), ('five', 6647), ('s', 6606), ('that', 6541), ('for', 6447), ('four', 6338), ('six', 6239), ('seven', 5914), ('with', 5672), ('on', 5273), ('his', 4573), ('are', 4484), ('from', 4405), ('or', 4397), ('it', 4370), ('an', 4115), ('he', 3966), ('be', 3420), ('which', 3398), ('at', 3288), ('this', 3286), ('not', 2635), ('also', 2548), ('have', 2429), ('but', 2200), ('has', 2108), ('were', 2081), ('other', 1914), ('had', 1799), ('their', 1785), ('its', 1771), ('first', 1764), ('american', 1760), ('they', 1696), ('some', 1670)]


In [16]:
def vizinhos(word, k=10):
    sims = emb @ emb[word2idx[word]]
    top = sims.topk(k + 1).indices[1:].tolist()
    return [(str(vocab[i]), round(sims[i].item(), 3)) for i in top]

for word in ["nine", "he", "war", "anarchism", "government"]:
    print(word, vizinhos(word))

nine [('one', 0.769), ('gollancz', 0.728), ('eight', 0.728), ('okocha', 0.717), ('yurievsky', 0.713), ('helguson', 0.71), ('metamorphoses', 0.702), ('paperback', 0.701), ('pratchett', 0.699), ('matt', 0.698)]
he [('she', 0.746), ('speer', 0.613), ('crowley', 0.596), ('ashoka', 0.595), ('schopenhauer', 0.577), ('sedgwick', 0.572), ('agassi', 0.565), ('agrippina', 0.563), ('aristotle', 0.56), ('eventually', 0.556)]
war [('wars', 0.629), ('civil', 0.599), ('invasion', 0.582), ('fighting', 0.555), ('occupation', 0.545), ('conflict', 0.54), ('veterans', 0.537), ('revolution', 0.532), ('castigo', 0.525), ('kuwait', 0.524)]
anarchism [('geolibertarianism', 0.833), ('anarchist', 0.771), ('individualist', 0.751), ('georgism', 0.742), ('insurrectionary', 0.739), ('bureaucrash', 0.732), ('capitalism', 0.707), ('nihilist', 0.704), ('anarchists', 0.702), ('laissez', 0.662)]
government [('territory', 0.588), ('parliament', 0.578), ('state', 0.577), ('federal', 0.576), ('constitution', 0.568), ('legi

In [23]:
print(word, vizinhos("king"))

government [('prince', 0.683), ('queen', 0.629), ('son', 0.609), ('duke', 0.605), ('pope', 0.601), ('iv', 0.595), ('brother', 0.589), ('emperor', 0.577), ('princess', 0.572), ('margrave', 0.568)]


In [26]:
def analogia(a, b, c, k=5):
    v = F.normalize(emb[word2idx[b]] - emb[word2idx[a]] + emb[word2idx[c]], dim=0)
    sims = emb @ v
    for w in (a, b, c):
        sims[word2idx[w]] = -1
    return [str(vocab[i]) for i in sims.topk(k).indices.tolist()]

print(analogia("man", "king", "woman"))


['son', 'prince', 'daughter', 'elder', 'wife']


In [27]:
analogias = [
    ("man", "king", "woman", "queen"),
    ("man", "woman", "brother", "sister"),
    ("man", "woman", "he", "she"),
    ("he", "his", "she", "her"),
    ("france", "paris", "germany", "berlin"),
    ("france", "paris", "italy", "rome"),
    ("france", "french", "germany", "german"),
    ("good", "better", "bad", "worse"),
    ("big", "bigger", "small", "smaller"),
    ("one", "two", "three", "four"),
]

print("| Analogia | Esperado | Top 5 | Acertou (top 1 / top 5) |")
print("|---|---|---|---|")
for a, b, c, esperado in analogias:
    faltando = [w for w in (a, b, c) if w not in word2idx]
    if faltando:
        print(f"| `{a} : {b} :: {c} : ?` | {esperado} | fora do vocabulário: {', '.join(faltando)} | - |")
        continue
    top = analogia(a, b, c)
    acerto = f"{'sim' if top[0] == esperado else 'não'} / {'sim' if esperado in top else 'não'}"
    print(f"| `{a} : {b} :: {c} : ?` | {esperado} | {', '.join(top)} | {acerto} |")

| Analogia | Esperado | Top 5 | Acertou (top 1 / top 5) |
|---|---|---|---|
| `man : king :: woman : ?` | queen | son, prince, daughter, elder, wife | não / não |
| `man : woman :: brother : ?` | sister | wife, mother, son, daughter, cousin | não / não |
| `man : woman :: he : ?` | she | she, speer, himself, agrippina, augustus | sim / sim |
| `he : his :: she : ?` | her | her, my, achilles, gemellus, husband | sim / sim |
| `france : paris :: germany : ?` | berlin | berlin, vienna, moscow, doubleday, again | sim / sim |
| `france : paris :: italy : ?` | rome | berlin, moscow, vienna, gaul, kentucky | não / não |
| `france : french :: germany : ?` | german | german, polish, russian, spanish, swedish | sim / sim |
| `good : better :: bad : ?` | worse | deutelbaum, slower, undigested, bigger, locally | não / não |
| `big : bigger :: small : ?` | smaller | smaller, distant, larger, specialized, substantially | sim / sim |
| `one : two :: three : ?` | four | four, five, six, idh, vwls | si

## Resultados

| Tokens | Vocabulário | Pares (janela 2) | Épocas | Tempo de execução |
|---|---|---|---|---|
| 1.000.000 | 52.754 | 3.999.994 | 5 | 5h47min03s (~1h09 por época) |

### Analogias

| Analogia | Esperado | Top 5 | Acertou (top 1 / top 5) |
|---|---|---|---|
| `man : king :: woman : ?` | queen | son, prince, daughter, elder, wife | não / não |
| `man : woman :: brother : ?` | sister | wife, mother, son, daughter, cousin | não / não |
| `man : woman :: he : ?` | she | she, speer, himself, agrippina, augustus | sim / sim |
| `he : his :: she : ?` | her | her, my, achilles, gemellus, husband | sim / sim |
| `france : paris :: germany : ?` | berlin | berlin, vienna, moscow, doubleday, again | sim / sim |
| `france : paris :: italy : ?` | rome | berlin, moscow, vienna, gaul, kentucky | não / não |
| `france : french :: germany : ?` | german | german, polish, russian, spanish, swedish | sim / sim |
| `good : better :: bad : ?` | worse | deutelbaum, slower, undigested, bigger, locally | não / não |
| `big : bigger :: small : ?` | smaller | smaller, distant, larger, specialized, substantially | sim / sim |
| `one : two :: three : ?` | four | four, five, six, idh, vwls | sim / sim |

Acertos: 6 de 10, todos já no top 1.